# NMF with TIDF and Evaluation Metrics
* TF-IDF for vectorization (downweigh overly common words)
* NMF for topic modeling
* Evaluate model quality across different k using coherence, diversity and reconstruction error <br>
--> Plot metrics
* Visualize topic space using PCA & UMAP <br>
--> Present topic clusters and top words for optimal k

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load preprocessed data
df = pd.read_pickle("arxiv_short.pkl")
print(df.shape)
df.head()

In [ ]:
# Join tokens into text for TF–IDF
documents = df['tokens'].apply(lambda x: ' '.join(x))

# TF–IDF vectorization
vectorizer = TfidfVectorizer(max_features=100000)
X = vectorizer.fit_transform(documents)
feature_names = vectorizer.get_feature_names_out()

### Define Helper Functions for Evaluation Metrics & Visualization

In [ ]:
# Helper functions
def get_top_words(model, feature_names, n_top=15):
    """Return top words for each topic as list of lists."""
    topics = []
    for comp in model.components_:
        top_idx = np.argsort(comp)[::-1][:n_top]
        topics.append([feature_names[i] for i in top_idx])
    return topics

def topic_diversity(topics, topk=10):
    """Compute topic diversity: fraction of unique words among topk words."""
    top_words = [word for topic in topics for word in topic[:topk]]
    return len(set(top_words)) / len(top_words)

### NMF Loop for different values of *k*

In [ ]:
# Setup
topic_nums = range(2, 61, 1)
errors, coherences, diversities = [], [], []

tokenized_docs = df['tokens'].tolist()
dictionary = Dictionary(tokenized_docs)

In [ ]:
# Main loop for different values of k
for k in topic_nums:
    print(f"Running NMF for k={k}")

    # Fit NMF model
    nmf = NMF(n_components=k, random_state=42, init='nndsvda', max_iter=1000)
    W = nmf.fit_transform(X)
    H = nmf.components_
    errors.append(nmf.reconstruction_err_)

    # Get top words per topic
    topics = get_top_words(nmf, feature_names, n_top=15)

    # Assign dominant topic
    df_temp = df.copy()
    df_temp['dominant_topic'] = W.argmax(axis=1)

    # Topic labels for display
    topic_labels = {}
    for topic_idx, topic in enumerate(H):
        top_words = [feature_names[i] for i in topic.argsort()[:-4:-1]]
        topic_labels[topic_idx] = f"Topic {topic_idx}: {', '.join(top_words)}"

    df_temp['topic_label'] = df_temp['dominant_topic'].map(topic_labels)

    # Compute metrics
    # Topic coherence
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    coherences.append(coherence_model.get_coherence())
    # Topic diversity
    diversities.append(topic_diversity(topics))


### Plot Evaluation Metrics

In [ ]:
# Build summary table
summary = pd.DataFrame({
    'k': topic_nums,
    'Coherence': coherences,
    'Diversity': diversities,
    'Reconstruction_Error': errors
})

# Normalize metrics (0–1 scale)
scaler = MinMaxScaler()
summary[['Coherence_norm', 'Diversity_norm', 'Error_norm']] = scaler.fit_transform(
    summary[['Coherence', 'Diversity', 'Reconstruction_Error']]
)

# Calculate weighted composite score
weights = {'Coherence': 0.6, 'Diversity': 0.3, 'Reconstruction_Error': 0.1}
summary['Composite_Score'] = (
    weights['Coherence'] * summary['Coherence_norm'] +
    weights['Diversity'] * summary['Diversity_norm'] +
    weights['Reconstruction_Error'] * summary['Error_norm']
)

# Print table with scores
summary[['k', 'Coherence', 'Diversity', 'Reconstruction_Error', 'Coherence_norm', 'Diversity_norm', 'Error_norm']]

In [ ]:
# Plot model evaluation metrics
plt.figure(figsize=(12, 6))

plt.plot(summary['k'], summary['Coherence_norm'], marker='o', color='#1f77b4', label='Coherence (normalized)', alpha=0.8)
plt.plot(summary['k'], summary['Diversity_norm'], marker='o', color='#9467bd', label='Diversity (normalized)', alpha=0.8)
plt.plot(summary['k'], summary['Error_norm'], marker='o', color='#ff7f0e', label='Reconstruction Error (normalized)', alpha=0.8)

# Highlight best k
plt.axvline(11, color='darkred', linestyle='--', label='Optimum k=11', alpha=0.8)

plt.xlabel("Number of Topics (k)", fontsize=14)
plt.ylabel("Normalized Metric Score", fontsize=14)
plt.title("Normalized Model Evaluation Metrics", fontsize=16)
plt.legend(fontsize=12)
plt.grid(alpha=0.3)

# Axes & tick styling
ax = plt.gca()

for spine in ax.spines.values():
    spine.set_color(spine_color)

ax.tick_params(
    axis='both',
    which='both',
    bottom=True,
    left=True,
    color=spine_color,
    labelcolor='black',
    labelsize=11
)

plt.tight_layout()
plt.savefig('Visualizations/nmf_all_metrics.png', dpi=300)
plt.show()

## NMF with best *k*

In [ ]:
# Selected model with k=11
k = 11

nmf = NMF(n_components=k, random_state=42, init='nndsvda', max_iter=1000)
W = nmf.fit_transform(X)
H = nmf.components_

# Print top words per topic
num_top_words = 10
for topic_idx, topic in enumerate(H):
    top_indices = topic.argsort()[:-num_top_words - 1:-1]
    top_words = [feature_names[i] for i in top_indices]

# Get top words per topic
topics = get_top_words(nmf, feature_names, n_top=15)
pd.DataFrame(topics).to_csv(f"NMF_Topics/topic_words_k{k}.csv", index_label="Topic")

# Assign dominant topic
df_temp = df.copy()
df_temp['dominant_topic'] = W.argmax(axis=1)

In [ ]:
# Create topic labels for better plot readability/ interpretability
topic_labels = {
    0: "Deep learning & neural networks",
    1: "Condensed matter & magnetism",
    2: "Numerical optimization & SDEs",
    3: "Quantum computing",
    4: "GNNs & graph algorithms",
    5: "LLMs & NLP",
    6: "Computer vision & image processing",
    7: "Astrophysics – galaxy formation",
    8: "Robotics & human–machine interactions",
    9: "Linear algebra & mathematical theory",
    10: "Black hole & dark matter physics"
}

In [ ]:
# Visualize distribution of paper across topics
fig, ax = plt.subplots(figsize=(12, 6))

# Count documents per topic and sort ascending for horizontal bar
topic_counts = df_temp['dominant_topic'].value_counts().sort_values(ascending=True)
topics_sorted = topic_counts.index

# Wrap labels and append topic number
wrapped_labels = [
"\n".join(textwrap.wrap(f"{topic_labels[t]}", width=25))
for t in topics_sorted
]

# Horizontal bar plot
ax.barh(
    y=range(len(topics_sorted)),
    width=topic_counts.values,
    color="#1f77b4"
)

# y-Axis labels
ax.set_yticks(range(len(topics_sorted)))
ax.set_yticklabels(wrapped_labels, fontsize=11)

# Labels and title
ax.set_xlabel("Number of Publications", fontsize=14)
ax.set_ylabel("Topic Label", fontsize=14)
ax.set_title(f"Publications by Topic (k={k})", fontsize=16)

# Tick labels
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))
ax.tick_params(axis='x', colors='black', labelsize=11)

# Spines
ax.spines['top'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.grid(axis="y", visible=False)
plt.tight_layout()
plt.savefig(f'Visualizations/pubs_per_topic_n{k}.png', dpi=300)
plt.show()

### Projection into 2-D Space

In [ ]:
# Create color map for topics
topic_ids = sorted(df_temp['dominant_topic'].unique())
cmap = plt.colormaps.get_cmap('tab20')
colors = [cmap(i) for i in range(len(topic_ids))]
color_map = {t: colors[i % len(colors)] for i, t in enumerate(topic_ids)}
topic_colors = df_temp['dominant_topic'].map(color_map)

# PCA projection
pca = PCA(n_components=2, random_state=42)
W_pca = pca.fit_transform(W)

# UMAP projection
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
W_umap = reducer.fit_transform(W)

In [ ]:
# Plot Projections
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
titles = ["PCA Projection", "UMAP Projection"]

for ax, emb, title in zip(axes, [W_pca, W_umap], titles):
    ax.scatter(
        emb[:, 0], emb[:, 1],
        c=topic_colors,
        s=6,
        alpha=0.6
    )
    ax.set_title(f"{title} of NMF Topic Space (k={k})", fontsize=15)
    ax.set_xlabel("Component 1", fontsize=13)
    ax.set_ylabel("Component 2", fontsize=13)
    ax.grid(alpha=0.3)
    for spine in ax.spines.values():
        spine.set_color("#cccccc")
    ax.tick_params(axis='both', which='both', labelsize=10)

# Shared Legend
legend_elements = [
    Line2D(
        [0], [0],
        marker='o', color='w',
        markerfacecolor=color_map[t],
        markersize=8,
        label="\n".join(textwrap.wrap(topic_labels.get(t, f"Topic {t}"), width=18))
    )
    for t in topic_ids
    ]
axes[1].legend(
    handles=legend_elements,
    title="Topics",
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    fontsize=11,
    title_fontsize=13,
    ncol=1,
    frameon=False
)

# Axis ticks
for ax in axes:
    ax.tick_params(
        axis='both',
        which='both',
        bottom=True,
        left=True,
        color=spine_color,
        labelcolor='black'
    )

plt.tight_layout()
plt.savefig(f'Visualizations/nmf_pca_vs_umap_k{k}.png', dpi=300, bbox_inches='tight')
plt.show()